In [2]:
import pandas as pd
import itertools
import matplotlib.pyplot as plt
import numpy as np

# Generating Table

In [19]:
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    

models = list(model_params.keys())
all_configs = {}
for model in models:
    #print(f"Model: {model}")
    # model_variants = model_params[model].keys()
    # model_combinations = list(itertools.product(*(model_params[model][variant] for variant in model_variants)))
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    # print(f'model:{model} combinations:{model_combinations}')
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)



In [23]:
# for combination in total_combs: 

# read dataframe
res_df = pd.read_csv(f'./simulations/results/sim_distributions/{configuration}/nodes_10_to_29/n_graphs_3/res_df.csv', header=0, index_col=0)

# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      # 'divcurves_funs_diffs_shortest_path_distance': 'shortest path area between curves',
      'divcurves_shortest_path_distance': 'Shortest Path',
      # 'divcurves_mds_funs_diffs_shortest_path_distance': 'shortest path area between curves MDS',
      # 'divcurves_mds_shortest_path_distance': 'shortest path MDS',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      # 'kernel_subtree_wl': 'Kernel Subtree WL', 
      # 'kernel_odd_sth': 'Kernel ODD-STh', 
      # 'kernel_mds_odd_sth': 'Kernel ODD-STh MDS',
      # 'kernel_propagation': 'Kernel Propagation', 
      # 'kernel_mds_propagation': 'Kernel Propagation MDS',
      # 'kernel_vertex_histogram': 'Kernel Vertex Histogram', 
      'kernel_weisfeiler_lehman': 'Kernel WL',
      'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      'kernel_core_framework': 'Kernel Core Framework',
      'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
   #    'toper_degree_super',
      'toper_degree_sub': 'TopER Degree', 
   #    'toper_degree_sub_super', 
   #    'toper_popularity_super',
      'toper_popularity_sub': 'TopER Popularity', 
   #    'toper_popularity_sub_super',
   #    'toper_forricci_super', 
      'toper_forricci_sub': 'TopER Forricci',
   #    'toper_forricci_sub_super', 
   #    'toper_olricci_super', 
      'toper_olricci_sub': 'TopER Olricci',
   #    'toper_olricci_sub_super', 
   #    'toper_closeness_super',
      'toper_closeness_sub': 'TopER Closeness', 
   #    'toper_closeness_sub_super', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      # 'persistent_homology_degree': 'PH Degree',

      # 'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',

      # 'persistent_homology_forman_ricci_curvature': 'PH Fricci',

      # 'persistent_homology_laplacian': 'PH Laplacian', 

      # 'persistent_homology_vietoris_rips': 'PH Vietoris Rips',

      # 'persistent_homology_alpha_complex': 'PH Alpha Complex',

}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)


scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}


res_copy = df.copy()
#remove one scale
# res_copy.drop('div_one_scale_shortest_path_distance', axis=0, inplace=True)
# results_latex = pd.DataFrame(index=res_copy.index)
results_latex = {}


for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\\pm $ ' + '{:.3f}'.format(round(s, 3))
      )
      # best_mask = res_copy[mean_col] == res_copy[mean_col].max()
      # results_latex.loc[best_mask, (model, score)] = results_latex.loc[best_mask, (model, score)].apply(
      #     lambda x: f"\\textbf{{{x}}}"
      # )

results_latex_df = pd.DataFrame.from_dict(results_latex)


# Bold the best mean for each score within each top-level group

for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
 &  & Accuracy & Silhouette Score & Davies-Bouldin & Calinski-Harabasz & ARI k-means & ARI agglomerative \\
 & Method &  &  &  &  &  &  \\
\midrule
Diversity Curves & Shortest Path & \textbf{0.904 $\pm $ 0.059} & 0.423 $\pm $ 0.034 & 0.753 $\pm $ 0.060 & 191.237 $\pm $ 33.144 & 0.495 $\pm $ 0.071 & 0.313 $\pm $ 0.111 \\
 & Shortest Path PCA & 0.892 $\pm $ 0.065 & \textbf{0.432 $\pm $ 0.035} & \textbf{0.737 $\pm $ 0.060} & \textbf{194.064 $\pm $ 34.006} & \textbf{0.500 $\pm $ 0.075} & 0.323 $\pm $ 0.123 \\
Diversity Profile & Profile Shortest Path & 0.829 $\pm $ 0.029 & 0.033 $\pm $ 0.019 & 3.949 $\pm $ 0.390 & 12.517 $\pm $ 2.420 & 0.097 $\pm $ 0.050 & 0.095 $\pm $ 0.049 \\
Embeddings & NetLSD & 0.900 $\pm $ 0.033 & 0.055 $\pm $ 0.038 & 1.332 $\pm $ 0.112 & 38.799 $\pm $ 7.908 & 0.150 $\pm $ 0.054 & 0.092 $\pm $ 0.045 \\
 & FEATHE